#### Part 1: Exploration and Data Quality

In [8]:
# Read the raw data files and summarize them
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")

FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geo": "olist_geolocation_dataset.csv",
    "cat_en": "product_category_name_translation.csv",
}

d = {k: pd.read_csv(RAW / v) for k, v in FILES.items()}

summary = pd.DataFrame(
    [
        {
            "table": k,
            "filename": FILES[k],
            "rows": v.shape[0],
            "n_cols": v.shape[1],
            "columns": ", ".join(v.columns),
        }
        for k, v in d.items()
    ]
).set_index("table")

pd.set_option("display.max_colwidth", None)
summary

,filename,rows,n_cols,columns
table,,,,
orders,olist_orders_dataset.csv,99441,8,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date"
items,olist_order_items_dataset.csv,112650,7,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value"
payments,olist_order_payments_dataset.csv,103886,5,"order_id, payment_sequential, payment_type, payment_installments, payment_value"
reviews,olist_order_reviews_dataset.csv,99224,7,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp"
customers,olist_customers_dataset.csv,99441,5,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state"
sellers,olist_sellers_dataset.csv,3095,4,"seller_id, seller_zip_code_prefix, seller_city, seller_state"
products,olist_products_dataset.csv,32951,9,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm"
geo,olist_geolocation_dataset.csv,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state"
cat_en,product_category_name_translation.csv,71,2,"product_category_name, product_category_name_english"


##### The raw dataset consists of 9 tables:
1. Orders (99,441 rows × 8 columns): Contains order-level information, including order ID, customer ID, order status, and order timestamps.
2. Items (112,650 rows × 7 columns): Contains details of items in each order, including order ID, product ID, seller ID, shipping limit date, price, and freight value.
3. Payments (103,886 rows × 5 columns): Contains payment details for each order, including payment type, payment installments, payment sequence, and payment value.
4. Reviews (99,224 rows × 7 columns): Contains customer feedback for each order, including review score, review comments, and review timestamps.
5. Customers (99,441 rows × 5 columns): Contains customer identifiers and location details, including unique customer ID, ZIP code prefix, city, and state.
6. Sellers (3,095 rows × 4 columns): Contains seller identifiers and location details, including ZIP code prefix, city, and state.
7. Products (32,951 rows × 9 columns): Contains product attributes, including category, name and description lengths, number of photos, weight, and dimensions.
8. Geolocation (geo) (1,000,163 rows × 5 columns): Contains ZIP code prefixes and geographical information, including latitude, longitude, city, and state.
9. Category Translation (cat_en) (71 rows × 2 columns): Maps Portuguese product category names to their English equivalents for analysis.

In [ ]:
# This function is used to profile a dataframe and provide information about its shape
# Uniqueness of a specified key column, number of duplicated rows, data types, missing values, and unique values for each column.
def profile(df, key=None):
    print(f"shape: {df.shape}")
    if key:
        print(f"key {key!r} unique: {df[key].is_unique} "
              f"({df[key].nunique()} distinct of {len(df)} rows)")
    print(f"fully duplicated rows: {df.duplicated().sum()}")
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    return out

In [12]:
orders = d["orders"].copy()
profile(orders, key="order_id")

shape: (99441, 8)
key 'order_id' unique: True (99441 distinct of 99441 rows)
fully duplicated rows: 0


,dtype,n_missing,pct_missing,n_unique
order_id,str,0,0.00,99441
customer_id,str,0,0.00,99441
order_status,str,0,0.00,8
order_purchase_timestamp,str,0,0.00,98875
order_approved_at,str,160,0.16,90733
order_delivered_carrier_date,str,1783,1.79,81018
order_delivered_customer_date,str,2965,2.98,95664
order_estimated_delivery_date,str,0,0.00,459


**Oberservations of "orders table"**
- order_id is unique across all 99,441 rows, and there are no fully duplicated rows. So one row per order is a valid grain for the base table.
- Dtypes: pandas shows str. It means the same thing here: nothing is parsed as a date yet.
- Missing values. Approved has 160 missing, carrier handoff 1,783, and customer delivery 2,965. These follow the order lifecycle (purchase, then approval, then carrier handoff, then delivery), so a missing later stage usually means an earlier stage never completed, not that data was lost. That's the "event never happened" kind of missing the brief wants us to separate from collection gaps.
- order_purchase_timestamp has 98,875 distinct values for 99,441 rows. That's just several orders placed in the same second. It isn't a duplicate problem, because order_id is unique.
- order_estimated_delivery_date has only 459 distinct values. It's a date with no time of day, so we'll compare it to delivery dates at day level.
The 160 orders with no approval time matter for Part 2. Our prediction point is order approval, so an order that was never approved has no prediction moment. We'll decide what to do with them later.


In [ ]:
# Count of each order status
orders["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**What the status counts show?**
- Delivered is 96,478 (97.0%). The other 2,963 orders never reached the customer according to the data.
- The statuses are stages of the lifecycle: created → approved → invoiced → processing → shipped → delivered. Then there are two exits, canceled and unavailable.
- shipped is the biggest non-delivered group (1,107). Those left the warehouse but have no delivery record. Some may be genuinely lost or late, and some may be recent orders that were still in transit when the data was extracted. 
- A mismatch to explain. Non-delivered orders total 2,963, but 2,965 orders have no customer delivery date. If missing delivery date meant "not delivered", the numbers would match exactly. They differ by 2, so a few orders have a status that disagrees with their timestamps.

In [14]:
date_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
orders.groupby("order_status")[date_cols].apply(lambda g: g.isna().sum()).assign(
    n_orders=orders["order_status"].value_counts()
)

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,n_orders
order_status,,,,
approved,0,2,2,2
canceled,141,550,619,625
created,5,5,5,5
delivered,14,2,8,96478
invoiced,0,314,314,314
processing,0,301,301,301
shipped,0,0,1107,1107
unavailable,0,609,609,609


In [15]:
odd = {
    "delivered, no approval": (orders.order_status == "delivered") & orders.order_approved_at.isna(),
    "delivered, no carrier date": (orders.order_status == "delivered") & orders.order_delivered_carrier_date.isna(),
    "delivered, no delivery date": (orders.order_status == "delivered") & orders.order_delivered_customer_date.isna(),
    "canceled, has delivery date": (orders.order_status == "canceled") & orders.order_delivered_customer_date.notna(),
}
for name, mask in odd.items():
    print(name, "->", mask.sum())

orders[odd["delivered, no delivery date"] | odd["canceled, has delivery date"]].sort_values("order_status")

delivered, no approval -> 14
delivered, no carrier date -> 2
delivered, no delivery date -> 8
canceled, has delivery date -> 6


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
